# 01 Momentum Research — Macro Metals System

> **Strategy:** Time-Series Momentum (Memory File §3.1)
> **Scope:** In-sample development (2015–2022), daily rebalance
> **Instruments:** Gold, Silver futures · EURUSD, USDJPY, AUDUSD · SOFR front
> **Signals:** ROC(20/63/126/252) blend + EMA(8/32) crossover + Donchian breakout
>
> Self-contained BQuant notebook — executable top-to-bottom.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml
from pathlib import Path
from datetime import datetime

# Bloomberg BQL
import bql
bq = bql.Service()

print(f"Session started : {datetime.now():%Y-%m-%d %H:%M}")
print(f"BQL service     : {type(bq).__name__}")
print(f"NumPy {np.__version__}  |  pandas {pd.__version__}")

## Config & Parameters

Load `parameters.yaml` and `tickers.yaml` from `config/`.
All strategy knobs come from the YAML — no magic numbers in code.

In [ ]:
CONFIG_DIR = Path("../config")

with open(CONFIG_DIR / "parameters.yaml") as f:
    params = yaml.safe_load(f)
with open(CONFIG_DIR / "tickers.yaml") as f:
    tickers = yaml.safe_load(f)

gcfg    = params["global"]
mom_cfg = params["strategies"]["momentum"]
targets = params["performance_targets"]

# Flatten instrument universe from config groups
UNIVERSE = {}
for asset_class, keys in mom_cfg["instruments"].items():
    for k in keys:
        UNIVERSE[k] = asset_class

LABELS = {
    "gc_fut_front":   "Gold (GC)",
    "si_fut_front":   "Silver (SI)",
    "pl_fut_front":   "Platinum (PL)",
    "eurusd_spot":    "EURUSD",
    "usdjpy_spot":    "USDJPY",
    "audusd_spot":    "AUDUSD",
    "sofr_fut_front": "SOFR Front",
    "sofr_fut_second":"SOFR 2nd",
    "sofr_fut_third": "SOFR 3rd",
    "sofr_fut_fourth":"SOFR 4th",
}

IS_START = gcfg["in_sample_start"]
IS_END   = gcfg["in_sample_end"]

print("Momentum parameters:")
print(f"  Lookbacks      : {mom_cfg['lookbacks_days']}")
print(f"  Blend weights  : {mom_cfg['signal_blend_weights']}")
print(f"  EMA fast/slow  : {mom_cfg['ema_fast']}/{mom_cfg['ema_slow']}")
print(f"  Breakout ch.   : {mom_cfg['breakout_channel_days']}d")
print(f"  Signal cap     : {mom_cfg['signal_cap']}")
print(f"  Vol target     : {mom_cfg['vol_target_annual']:.0%}")
print(f"  Stop-loss      : {mom_cfg['stop_loss_multiple']}x vol")
print(f"  Cooldown       : {mom_cfg['cooldown_days_after_stop']}d")
print(f"  IS period      : {IS_START} to {IS_END}")
print(f"  Universe       : {len(UNIVERSE)} instruments")

## Data Pipeline (BQL)

Fetch daily `PX_LAST` for the full momentum universe via Bloomberg BQL.
Handle missing data and timeouts gracefully.

In [ ]:
class BQuantDataLoader:
    """Fetch historical prices via Bloomberg BQL.

    Resolves logical names from tickers.yaml, fetches PX_LAST over a
    date range, and returns clean pd.Series.
    """

    def __init__(self, ticker_map: dict) -> None:
        self._tickers = ticker_map
        self._bq = bql.Service()

    def resolve(self, logical_name: str) -> str:
        """Logical name -> Bloomberg ticker string."""
        for group in self._tickers.values():
            if isinstance(group, dict) and logical_name in group:
                return group[logical_name]
        raise KeyError(f"{logical_name} not in tickers.yaml")

    def get_history(
        self,
        logical_name: str,
        start: str,
        end: str,
        field: str = "PX_LAST",
    ) -> pd.Series:
        """Fetch daily price series for one instrument.

        Args:
            logical_name: Key in tickers.yaml (e.g. 'gc_fut_front').
            start / end:  ISO date strings.
            field:        Bloomberg field (default PX_LAST).

        Returns:
            pd.Series indexed by date, named after the logical key.
        """
        bbg = self.resolve(logical_name)
        request = bql.Request(
            bbg,
            {field: self._bq.data.px_last(
                dates=self._bq.func.range(start, end)
            )},
        )
        try:
            response = self._bq.execute(request)
            df = response[0].df()
            if df.empty:
                print(f"  ! {logical_name}: empty response")
                return pd.Series(dtype=float, name=logical_name)
            # BQL returns MultiIndex (DATE, ID) — drop ID level
            if isinstance(df.index, pd.MultiIndex):
                series = df[field].droplevel("ID")
            else:
                series = df[field]
            series = series.sort_index().astype(float)
            series.name = logical_name
            series.index.name = "date"
            return series
        except Exception as exc:
            print(f"  ! {logical_name} ({bbg}): {exc}")
            return pd.Series(dtype=float, name=logical_name)


# --- Fetch all instruments ---
loader = BQuantDataLoader(tickers)
prices = {}

for key in UNIVERSE:
    label = LABELS.get(key, key)
    print(f"  {label:20s}", end=" ")
    s = loader.get_history(key, IS_START, IS_END)
    if len(s) > 0:
        prices[key] = s
        print(f"OK  {len(s):>5d} obs  "
              f"[{s.index[0]:%Y-%m-%d} -> {s.index[-1]:%Y-%m-%d}]")
    else:
        print("MISSING")

prices_df = pd.DataFrame(prices).sort_index().ffill()
print(f"\nPanel: {prices_df.shape[0]} days x {prices_df.shape[1]} instruments")
prices_df.tail(3)

## Momentum Strategy

Three signal components blended into a composite score per instrument:

| Component | Weight | Description |
|-----------|--------|-------------|
| **ROC blend** | 60% | Rate-of-change at 20/63/126/252d, equal-weighted, tanh-normalised |
| **EMA crossover** | 25% | Sign of EMA(8) - EMA(32) |
| **Donchian breakout** | 15% | +1 at channel high, -1 at channel low |

Final signal clipped to `[-signal_cap, +signal_cap]` from config.

In [ ]:
class MomentumStrategy:
    """Time-series momentum from memory file section 3.1.

    Blends rate-of-change, EMA crossover, and Donchian breakout signals
    into a composite score in [-1, +1] per instrument.
    """

    # Component blend weights (documented in markdown above)
    ROC_WEIGHT      = 0.60
    EMA_WEIGHT      = 0.25
    BREAKOUT_WEIGHT = 0.15

    def __init__(self, params: dict) -> None:
        cfg = params["strategies"]["momentum"]
        self.lookbacks: list[int]   = cfg["lookbacks_days"]
        self.weights: list[float]   = cfg["signal_blend_weights"]
        self.ema_fast: int          = cfg["ema_fast"]
        self.ema_slow: int          = cfg["ema_slow"]
        self.breakout_days: int     = cfg["breakout_channel_days"]
        self.signal_cap: float      = cfg["signal_cap"]

    # ----- Signal components ------------------------------------------------

    def _roc_signal(self, price: pd.Series, window: int) -> pd.Series:
        """Rate-of-change normalised to [-1, +1] via tanh."""
        roc = price / price.shift(window) - 1
        # Scale by rolling vol for risk-adjusted signal
        vol = price.pct_change().rolling(window, min_periods=max(window // 2, 5)).std()
        vol = vol.replace(0, np.nan)
        return np.tanh(roc / vol)

    def _ema_signal(self, price: pd.Series) -> pd.Series:
        """EMA crossover: +1 when fast > slow, -1 otherwise."""
        fast = price.ewm(span=self.ema_fast, adjust=False).mean()
        slow = price.ewm(span=self.ema_slow, adjust=False).mean()
        return np.sign(fast - slow)

    def _breakout_signal(self, price: pd.Series) -> pd.Series:
        """Donchian channel breakout: +1 at new high, -1 at new low."""
        upper = price.rolling(self.breakout_days).max()
        lower = price.rolling(self.breakout_days).min()
        sig = pd.Series(0.0, index=price.index)
        sig[price >= upper] = 1.0
        sig[price <= lower] = -1.0
        # Persist last non-zero signal (hold until reversed)
        return sig.replace(0, np.nan).ffill().fillna(0.0)

    # ----- Composite --------------------------------------------------------

    def generate_signals(self, prices_df: pd.DataFrame) -> pd.DataFrame:
        """Generate composite momentum signals for all instruments.

        Args:
            prices_df: DataFrame of daily prices (columns = instruments).

        Returns:
            DataFrame with same shape, values in [-signal_cap, +signal_cap].
        """
        signals = pd.DataFrame(
            index=prices_df.index, columns=prices_df.columns, dtype=float
        )

        for col in prices_df.columns:
            price = prices_df[col].dropna()
            if len(price) < max(self.lookbacks) + 20:
                signals[col] = 0.0
                continue

            # 1) ROC blend across lookback windows
            roc_blend = pd.Series(0.0, index=price.index)
            for lb, w in zip(self.lookbacks, self.weights):
                roc_blend += w * self._roc_signal(price, lb)

            # 2) EMA crossover
            ema_sig = self._ema_signal(price)

            # 3) Donchian breakout
            brk_sig = self._breakout_signal(price)

            # Composite blend
            composite = (
                self.ROC_WEIGHT      * roc_blend
                + self.EMA_WEIGHT    * ema_sig
                + self.BREAKOUT_WEIGHT * brk_sig
            )

            signals[col] = composite.clip(
                -self.signal_cap, self.signal_cap
            )

        return signals.reindex(prices_df.index).ffill().fillna(0.0)

## Generate Signals (IS Period)

In [ ]:
momentum = MomentumStrategy(params)
signals = momentum.generate_signals(prices_df)

print(f"Signal matrix: {signals.shape[0]} days x {signals.shape[1]} instruments")
print(f"Date range:    {signals.index[0]:%Y-%m-%d} to {signals.index[-1]:%Y-%m-%d}")
print()

renamed = signals.rename(columns=LABELS)
print("Signal statistics:")
renamed.describe().round(3)

## Backtest Engine

Inline vectorised backtest per the memory file:
- Position = signal x (vol_target / instrument_vol), capped at 2x
- Transaction costs: 2 bp per side on notional turnover
- One-day execution lag (signal on day *t* drives position on *t+1*)

In [ ]:
def backtest_single_asset(
    prices: pd.Series,
    signals: pd.Series,
    vol_target: float = 0.10,
    vol_lookback: int = 30,
    vol_cap: float = 2.0,
    tc_bp: float = 2.0,
    capital: float = 1_000_000.0,
) -> pd.DataFrame:
    """Vectorised single-asset backtest.

    Args:
        prices:      Daily prices.
        signals:     Raw signals in [-1, +1], lagged by 1 day internally.
        vol_target:  Annualised vol target (default 10%).
        vol_lookback: EWMA halflife in days.
        vol_cap:     Max position scaling factor.
        tc_bp:       Transaction cost per side in basis points.
        capital:     Starting notional.

    Returns:
        DataFrame: position, price, ret_gross, ret_net, equity, turnover.
    """
    ret = prices.pct_change().fillna(0.0)

    # EWMA annualised vol
    ann_vol = ret.ewm(halflife=vol_lookback).std() * np.sqrt(252)
    ann_vol = ann_vol.replace(0, np.nan)
    vol_scale = (vol_target / ann_vol).clip(upper=vol_cap).fillna(1.0)

    # Lag signal, apply vol scaling
    position = (signals.shift(1).fillna(0.0) * vol_scale).clip(-vol_cap, vol_cap)

    # PnL
    ret_gross = position * ret
    turnover  = position.diff().abs().fillna(0.0)
    cost      = turnover * (tc_bp / 10_000)
    ret_net   = ret_gross - cost
    equity    = capital * (1 + ret_net).cumprod()

    return pd.DataFrame({
        "position": position,
        "price":    prices,
        "ret_gross": ret_gross,
        "ret_net":  ret_net,
        "equity":   equity,
        "turnover": turnover,
    })


def compute_metrics(bt: pd.DataFrame, periods: int = 252) -> dict:
    """Performance metrics consistent with memory file section 6.3."""
    r  = bt["ret_net"].dropna()
    eq = bt["equity"].dropna()

    n_years  = len(r) / periods
    total    = (1 + r).prod()
    ann_ret  = total ** (1 / max(n_years, 0.01)) - 1
    ann_vol  = r.std() * np.sqrt(periods)
    sharpe   = ann_ret / ann_vol if ann_vol > 0 else 0.0

    running_max = eq.cummax()
    dd = (eq - running_max) / running_max
    max_dd = float(-dd.min())

    calmar  = ann_ret / max_dd if max_dd > 0 else 0.0
    hit     = float((r > 0).sum() / len(r)) if len(r) > 0 else 0.0
    ann_to  = bt["turnover"].sum() / max(n_years, 0.01)

    return {
        "Ann. Return":  ann_ret,
        "Ann. Vol":     ann_vol,
        "Sharpe":       sharpe,
        "Max DD":       max_dd,
        "Calmar":       calmar,
        "Hit Rate":     hit,
        "Ann. Turnover": ann_to,
    }


print("Backtest engine ready.")

## Results

Run the backtest across all instruments, compute per-asset metrics,
and plot equity curves.

In [ ]:
results = {}
metrics_rows = []

for inst in prices_df.columns:
    bt = backtest_single_asset(
        prices_df[inst],
        signals[inst],
        vol_target=mom_cfg["vol_target_annual"],
        vol_lookback=gcfg["vol_lookback_days"],
        vol_cap=gcfg["vol_cap_multiplier"],
        tc_bp=2.0,
    )
    results[inst] = bt
    m = compute_metrics(bt)
    m["Instrument"] = LABELS.get(inst, inst)
    metrics_rows.append(m)

metrics_df = pd.DataFrame(metrics_rows).set_index("Instrument")

# Format for display
fmt_df = metrics_df.copy()
for col in ["Ann. Return", "Ann. Vol", "Max DD", "Hit Rate"]:
    fmt_df[col] = fmt_df[col].map("{:.1%}".format)
fmt_df["Sharpe"]  = fmt_df["Sharpe"].map("{:.2f}".format)
fmt_df["Calmar"]  = fmt_df["Calmar"].map("{:.2f}".format)
fmt_df["Ann. Turnover"] = fmt_df["Ann. Turnover"].map("{:.1f}x".format)

# Equity curves (normalised to 1.0)
equity_df = pd.DataFrame({
    LABELS.get(k, k): v["equity"] for k, v in results.items()
})
equity_norm = equity_df / equity_df.iloc[0]

fig_eq = px.line(
    equity_norm,
    title="Momentum Strategy — Equity Curves (IS: 2015-2022, $1M start)",
    labels={"value": "Growth of $1", "variable": "Instrument", "date": ""},
)
fig_eq.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.15),
    height=500,
    yaxis_tickformat="$.2f",
)
fig_eq.show()

print("Per-instrument metrics (IS period, 2 bp TC):")
fmt_df

## Visualisations

1. Gold (GC) price with signal overlay
2. Cross-instrument signal correlation heatmap

In [ ]:
# --- 1. GC signal overlay on price ---
gc_key = "gc_fut_front"
gc_bt  = results[gc_key]
gc_sig = signals[gc_key]

fig_gc = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=["Gold (GC) — Price", "Momentum Signal"],
    vertical_spacing=0.08,
)

fig_gc.add_trace(
    go.Scatter(
        x=gc_bt.index, y=gc_bt["price"],
        name="GC Price",
        line=dict(color="#FFD700", width=1.5),
    ),
    row=1, col=1,
)
fig_gc.add_trace(
    go.Bar(
        x=gc_sig.index, y=gc_sig.values,
        name="Signal",
        marker_color=[
            "#2ecc71" if v > 0 else "#e74c3c" for v in gc_sig.values
        ],
        opacity=0.7,
    ),
    row=2, col=1,
)
fig_gc.update_layout(
    template="plotly_white",
    height=600,
    showlegend=False,
    title_text="Gold Momentum — Price & Signal (IS)",
)
fig_gc.update_yaxes(title_text="USD/oz", row=1, col=1)
fig_gc.update_yaxes(title_text="Signal", range=[-1.1, 1.1], row=2, col=1)
fig_gc.show()

# --- 2. Signal correlation heatmap ---
corr = signals.rename(columns=LABELS).corr()
fig_corr = px.imshow(
    corr.round(2),
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Momentum Signal Correlation Matrix",
    aspect="equal",
)
fig_corr.update_layout(template="plotly_white", height=550, width=650)
fig_corr.show()

## Performance Summary

Compare portfolio-average metrics against the memory file targets (section 6.3).
Per-strategy Sharpe target is **> 0.5**.

In [ ]:
# Aggregate across instruments
avg = metrics_df.mean()

comparison = pd.DataFrame({
    "Metric": [
        "Sharpe Ratio",
        "Annualised Vol",
        "Max Drawdown",
        "Calmar Ratio",
        "Hit Rate",
        "Ann. Turnover",
    ],
    "Target (section 6.3)": [
        f"> {targets['sharpe_per_strategy']:.1f}",
        f"{targets['vol_range_annual'][0]:.0%} - {targets['vol_range_annual'][1]:.0%}",
        f"< {targets['max_drawdown_pct']:.0f}%",
        f"> {targets['calmar_ratio']:.1f}",
        f"> {targets['hit_rate_daily']:.0%}",
        f"< {targets['max_turnover_annual']:.0f}x",
    ],
    "Portfolio Average": [
        f"{avg['Sharpe']:.2f}",
        f"{avg['Ann. Vol']:.1%}",
        f"{metrics_df['Max DD'].max():.1%}",
        f"{avg['Calmar']:.2f}",
        f"{avg['Hit Rate']:.1%}",
        f"{avg['Ann. Turnover']:.1f}x",
    ],
}).set_index("Metric")

print("=" * 65)
print("  MOMENTUM STRATEGY vs MEMORY FILE TARGETS  (IS: 2015-2022)")
print("=" * 65)
comparison

## Export

Save signals, equity curves, and summary table to `outputs/`.

In [ ]:
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

datestamp = datetime.now().strftime("%Y%m%d")

# Signals
sig_path = output_dir / f"momentum_signals_is_{datestamp}.csv"
signals.rename(columns=LABELS).to_csv(sig_path)

# Equity curves
eq_path = output_dir / f"momentum_equity_is_{datestamp}.csv"
equity_df.to_csv(eq_path)

# Summary HTML
html_path = output_dir / f"momentum_summary_{datestamp}.html"
html_content = (
    "<h2>Momentum Strategy — IS Performance (2015-2022)</h2>\n"
    + fmt_df.to_html()
    + "<br><h3>vs Memory File Targets</h3>\n"
    + comparison.to_html()
)
with open(html_path, "w") as f:
    f.write(html_content)

print(f"Exported to {output_dir.resolve()}/")
print(f"  {sig_path.name:45s}  ({signals.shape[0]} rows x {signals.shape[1]} cols)")
print(f"  {eq_path.name:45s}  ({equity_df.shape[0]} rows)")
print(f"  {html_path.name}")
print(f"\nNotebook complete: {datetime.now():%Y-%m-%d %H:%M}")